## Building A Chatbot
An example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that we may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This example will cover the basics which will be helpful for those two more advanced topics.

In [8]:
import os
from dotenv import load_dotenv

load_dotenv("../../.env")

groq_api_key = os.getenv("GROQ_API_KEY")

In [9]:
from langchain_groq import ChatGroq

model = ChatGroq(model="llama-3.1-8b-instant", groq_api_key=groq_api_key) #llama-3.1-8b-instant
model
 

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001954686AAD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001954686BB10>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [10]:
from langchain_core.messages import HumanMessage

model.invoke([HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")])


AIMessage(content="Hello Vaibhav, nice to meet you. I'm glad to hear you're a software engineer. How can I assist you today? Are you working on a specific project or do you have any questions about software development?", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 52, 'total_tokens': 99, 'completion_time': 0.067276086, 'completion_tokens_details': None, 'prompt_time': 0.126468716, 'prompt_tokens_details': None, 'queue_time': 0.227749109, 'total_time': 0.193744802}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d86f5-b4b5-7d12-b4a0-da0a9f5f3f93-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 52, 'output_tokens': 47, 'total_tokens': 99})

In [11]:
from langchain_core.messages import AIMessage

model.invoke([
    HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer."),
    AIMessage(content="Hello Vaibhav! It's great to meet you. How can I assist you today?"),
    HumanMessage(content="Hey, what is my name and what do I do?")
])

AIMessage(content="Your name is Vaibhav, and you're a software engineer.", additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 94, 'total_tokens': 110, 'completion_time': 0.013714633, 'completion_tokens_details': None, 'prompt_time': 0.005224228, 'prompt_tokens_details': None, 'queue_time': 0.158136416, 'total_time': 0.018938861}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_7ccc667439', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d86ff-02b2-7533-b3fd-700875d1204e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 94, 'output_tokens': 16, 'total_tokens': 110})

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [16]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables import RunnableWithMessageHistory

store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

with_message_history = RunnableWithMessageHistory(model, get_session_history)

In [17]:
config = {"configurable":
            {"session_id": "chat1"}
    }

In [18]:
response = with_message_history.invoke(
    [HumanMessage(content="Hi, my name is Vaibhav, and I am a software engineer.")],
    config=config
)   

In [20]:
response.content

"Hello Vaibhav, nice to meet you. It's great to know that you are a software engineer. What type of projects or technologies are you currently working on or interested in?"

In [22]:
response = with_message_history.invoke(
    [HumanMessage(content="What is my name")],
    config=config
)
response.content

'Your name is Vaibhav.'

In [23]:
# Change the config =>>>>> change the config

config1 = {"configurable":
            {"session_id": "chat2"}
        }
response = with_message_history.invoke(
    [HumanMessage(content="What is my name?")]
    ,config=config1
)
response.content

"I don't have any information about your name. I'm a large language model, I don't have the ability to retain personal information or recall previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name, I can use it to address you, but it won't be stored or remembered in any way."